# review2: the Colab GPU session of 11 Nov 2026

The working notebook from the first GPU runs of the version-2 plan (Colab, Tesla T4), imported from Google Drive (`Untitled0.ipynb`) on 20 Nov 2026. **It is a session log, not a reproduction notebook**: the outputs are kept exactly as they ran. Colab had inserted cells out of order, so they are put back in the order they ran, read from the timestamps in their outputs; two empty cells were dropped.

**What it shows, in order:** setup on the GPU, timing tests, the first launch (one run ran out of memory), the runtime being reset (all results lost), a relaunch with saving after every seed, the one completed result (a credit comparison that exposed a bug), Google Drive mounted, and the bigger credit run started on the fixed code.

**What it does not contain:** any finished critic-budget result (lost to the reset, then the GPU quota ran out), the bigger credit run's results (partly on Drive, now void), the 10 Nov gate runs (`colab_runner.ipynb`), or the 19 Nov results (run on CPU; see `PROJECT_LOG.md`).

**Status after the 19 Nov review:** every number here was made before the critic-loss fix (commit `d812569`). Read it as history. The reruns use `tools/colab_resume.sh`.

## 1. Setup on the GPU (10:34 UTC)

Clones branch `set-critic` at commit `deb02e2`, installs the package and Jumanji, and checks JAX sees the GPU.

**Output:** a Tesla T4 with 15 GB. The pip conflict (`ibis-framework` / `toolz`) and the `jax_cuda13_plugin` traceback are warnings from Colab's preinstalled packages. The last line, `[CudaDevice(id=0)]`, is what matters: JAX runs on the GPU.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q -b set-critic https://github.com/yashwanthsagar2004-dot/accountable-teams
%cd /content/accountable-teams
!git log --oneline -1
!pip -q install -e . jumanji 2>&1 | tail -2
!python -c "import jax, jumanji; print(jax.__version__, jax.devices())"

name, memory.total [MiB]
Tesla T4, 15360 MiB
/content/accountable-teams
deb02e2 (HEAD -> set-critic, origin/set-critic) Log decision after AgenTracer and Aegis reviews
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.
/usr/local/lib/python3.13/dist-packages/jaxlib/plugin_support.py:99: RuntimeWarning: JAX plugin jax_cuda13_plugin version 0.11.1 is installed, but it is not compatible with the installed jaxlib version 0.12.0.1, so it will not be used.
  warnings.warn(
Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda13.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/jax/_src/xla_bridge.py", line 520, in discover_pjrt_plugins
    plugin_module.initialize()
    ~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/l

## 2. Timing tests (about 10:37 UTC)

Both experiment scripts at tiny sizes (20 updates; 5,000 decisions), only to check they run end to end and to time them.

**Output:** the numbers are not results. Reward 0.0 and the critic table with `+/- 0.000` (one seed) are what tiny runs give. `real 1m34s` and `1m51s` include JAX compilation. Each row of the table is an estimator, `[branches per decision]`, then relative error against ground truth (1.0 = as bad as predicting zero).

In [2]:
!time python experiments/credit_compare.py --env lbf --updates 20 --seeds 1 --out /tmp/cc_smoke.json 2>&1 | grep -v -i "plugin\|triton\|traceback\|File \|\^\|warn" | tail -8
!time python experiments/critic_budget.py --env lbf --decisions 5000 --train-steps 500 --seeds 1 --eval 10 --reps 20 --audit-reps 8 --out /tmp/cb_smoke.json 2>&1 | grep -v -i "plugin\|triton\|traceback\|File \|\^\|warn" | tail -12

{"credit": "team", "seed": 0, "final_reward_per_step": 0.0, "auc": 9.765625291038304e-05, "branches": 0, "branch_env_steps": 0, "env_steps": 20480, "seconds": 15.2}
{"credit": "resp", "seed": 0, "final_reward_per_step": 0.0, "auc": 4.882812645519152e-05, "branches": 5120, "branch_env_steps": 51200, "env_steps": 20480, "seconds": 42.7}
{"credit": "dr", "seed": 0, "final_reward_per_step": 0.0, "auc": 9.765625291038304e-05, "branches": 0, "branch_env_steps": 0, "env_steps": 20480, "seconds": 20.0}
team  final reward/step 0.0000 [0.0000, 0.0000]
resp  final reward/step 0.0000 [0.0000, 0.0000]
dr    final reward/step 0.0000 [0.0000, 0.0000]

real	1m34.362s
user	1m55.357s
sys	0m9.854s
  gated10        [   3.2] 1.283 +/- 0.000
  mc8            [   8.0] 2.282 +/- 0.000
  gated25        [   8.0] 1.283 +/- 0.000
  mc16           [  16.0] 1.676 +/- 0.000
  gated50        [  16.0] 1.394 +/- 0.000
  pw16           [  16.0] 1.105 +/- 0.000
  mc32           [  32.0] 1.320 +/- 0.000
  pw32           [

## 3. First launch of the full runs (10:41 UTC)

Starts three background jobs, each with 30% of GPU memory: the critic-budget comparison on Level-Based Foraging (window 20) and on RobotWarehouse (window 10), and the credit comparison (team vs responsibility vs counterfactual, 2,000 updates, 3 seeds).

**Output:** 13.7 GB of GPU memory taken, so the three jobs started.

In [3]:
%%bash
mkdir -p /content/runs && cd /content/accountable-teams
export XLA_PYTHON_CLIENT_MEM_FRACTION=0.3
nohup python experiments/critic_budget.py --env lbf --agents 4 --window 20 --seeds 3 --out /content/runs/critic_budget_lbf_w20.json > /content/runs/cb_lbf.log 2>&1 &
nohup python experiments/critic_budget.py --env rware --agents 4 --window 10 --seeds 3 --out /content/runs/critic_budget_rware_w10.json > /content/runs/cb_rware.log 2>&1 &
nohup python experiments/credit_compare.py --env lbf --agents 4 --updates 2000 --seeds 3 --out /content/runs/credit_compare_lbf.json > /content/runs/cc_lbf.log 2>&1 &
sleep 5; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv; date

utilization.gpu [%], memory.used [MiB]
0 %, 13755 MiB
Wed Nov 11 10:41:20 AM UTC 2026


## 4. First check (10:41 UTC)

**Output:** five matching processes, and empty logs, because the scripts print only when a seed finishes.

In [4]:
!date; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader; ps aux | grep -c "experiments/"; for f in /content/runs/*.log; do echo "== $f"; grep -v -i "plugin\|triton\|traceback\|File \|\^\|warn\|initialize" $f | tail -3; done

Wed Nov 11 10:41:31 AM UTC 2026
0 %, 13773 MiB
5
== /content/runs/cb_lbf.log
== /content/runs/cb_rware.log
== /content/runs/cc_lbf.log


## 5. RobotWarehouse runs out of memory (10:45 UTC)

**Output:** `RESOURCE_EXHAUSTED: Out of memory while trying to allocate 457.76MiB`. RobotWarehouse features are about ten times larger than LBF's, and 30% of the GPU was not enough. The two `credit: team` lines show the credit run finishing team-credit seeds 0 and 1.

In [7]:
!date; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader; tail -c 700 /content/runs/cb_rware.log; echo; tail -c 300 /content/runs/cb_lbf.log; echo; grep -o -e credit.....[a-z]* -e seconds.......... /content/runs/cc_lbf.log

Wed Nov 11 10:45:29 AM UTC 2026
100 %, 9317 MiB
/jax/_src/array.py", line 448, in __array__
    return np.asarray(self._value, dtype=dtype, **kwds)  # pyrefly: ignore[no-matching-overload]
                      ^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/jax/_src/profiler.py", line 588, in wrapper
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/jax/_src/array.py", line 696, in _value
    npy_value, did_copy = self._to_np_array_did_copy()
                          ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
jax.errors.JaxRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 457.76MiB with allocator GPU_0_bfc on device 0. [tf-allocator-allocation-error=''] [executable_name='jit_reshape']

^^^^^^^^^^^^^^^
AttributeError: module 'jax._src.lib.triton' has no attribute 'register_compilation_handler'
W1111 10:45:12.604624    4508 cuda_timer.cc:88] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup 

## 6. Requeue RobotWarehouse with more memory (10:46 UTC)

Starts a watcher that waits for the LBF critic job (process 4431) to finish, then runs RobotWarehouse with 60% of GPU memory.

**Output:** `lbf pid 4431` and a watcher running.

In [8]:
%%bash
cd /content/accountable-teams
P=$(pgrep -f critic_budget_lbf_w20 | head -1); echo lbf pid $P
nohup bash -c "while kill -0 $P 2>/dev/null; do sleep 30; done; XLA_PYTHON_CLIENT_MEM_FRACTION=0.6 python experiments/critic_budget.py --env rware --agents 4 --window 10 --seeds 3 --out /content/runs/critic_budget_rware_w10.json" > /content/runs/cb_rware.log 2>&1 &
sleep 2; ps aux | grep -c kill

lbf pid 4431
2


## 7. The runtime was reset (11:47 UTC)

**Output:** `up 2 min`, `0 MiB` of GPU memory, and `/content/runs` missing. Colab replaced the machine, and every running job and result was lost. The scripts saved only when a whole run finished. This led to commit `0aac712` (save after every seed) and later `0e50369` (save after every stage inside a seed).

In [1]:
!date; uptime; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader; ls -la /content/runs; pgrep -af experiments; grep -h -o -e credit.....[a-z]*.......seed.... -e seconds.......... -e RESOURCE_EXHAUSTED /content/runs/*.log; grep -h -A13 relative /content/runs/cb_lbf.log

Wed Nov 11 11:47:54 AM UTC 2026
 11:47:54 up 2 min,  0 user,  load average: 0.63, 0.36, 0.14
0 %, 0 MiB
ls: cannot access '/content/runs': No such file or directory
549 /bin/bash -c date; uptime; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader; ls -la /content/runs; pgrep -af experiments; grep -h -o -e credit.....[a-z]*.......seed.... -e seconds.......... -e RESOURCE_EXHAUSTED /content/runs/*.log; grep -h -A13 relative /content/runs/cb_lbf.log
grep: /content/runs/*.log: No such file or directory
grep: /content/runs/cb_lbf.log: No such file or directory


## 8. Relaunch with per-seed saving (11:54 UTC)

Fresh clone at commit `0aac712`, then the same three jobs (RobotWarehouse queued behind LBF).

**Output:** the commit line, the GPU at 9.2 GB, and three processes (2130 LBF critic, 2131 credit, 2132 the RobotWarehouse watcher).

In [2]:
%%bash
cd /content && rm -rf accountable-teams && git clone -q -b set-critic https://github.com/yashwanthsagar2004-dot/accountable-teams && cd accountable-teams && git log --oneline -1
pip -q install -e . jumanji > /tmp/pip.log 2>&1; tail -1 /tmp/pip.log
mkdir -p /content/runs
export XLA_PYTHON_CLIENT_MEM_FRACTION=0.3
nohup python experiments/critic_budget.py --env lbf --agents 4 --window 20 --seeds 3 --out /content/runs/critic_budget_lbf_w20.json > /content/runs/cb_lbf.log 2>&1 &
P=$!
nohup python experiments/credit_compare.py --env lbf --agents 4 --updates 2000 --seeds 3 --out /content/runs/credit_compare_lbf.json > /content/runs/cc_lbf.log 2>&1 &
nohup bash -c "while kill -0 $P 2>/dev/null; do sleep 30; done; XLA_PYTHON_CLIENT_MEM_FRACTION=0.6 python experiments/critic_budget.py --env rware --agents 4 --window 10 --seeds 3 --out /content/runs/critic_budget_rware_w10.json" > /content/runs/cb_rware.log 2>&1 &
sleep 20; date; nvidia-smi --query-gpu=name,utilization.gpu,memory.used --format=csv,noheader; pgrep -af experiments | cut -c1-90

0aac712 Checkpoint per seed and resume in critic_budget and credit_compare
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.
Wed Nov 11 11:54:22 AM UTC 2026
Tesla T4, 0 %, 9183 MiB
2130 python3 experiments/critic_budget.py --env lbf --agents 4 --window 20 --seeds 3 --out
2131 python3 experiments/credit_compare.py --env lbf --agents 4 --updates 2000 --seeds 3 -
2132 bash -c while kill -0 2130 2>/dev/null; do sleep 30; done; XLA_PYTHON_CLIENT_MEM_FRAC


## 9. The credit comparison finishes (12:29 UTC)

**Output:** the runtime survived (`up 44 min`). `credit_compare_lbf.json` was written. The extracted lines show **team credit 0.0073 reward per step on all three seeds, but responsibility credit 0.0 on every seed.** The lone `1.30208` is a fragment of a small number such as 1.3e-05, cut off by the `grep` pattern, not a reward of 1.3. The LBF critic job (2130) was still on its first seed.

In [3]:
!date; uptime; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader; ls -la /content/runs; pgrep -af experiments | cut -c1-80; grep -h -o -e credit.....[a-z]*.......seed.... -e final_reward_per_step.......... -e seconds.......... -e RESOURCE_EXHAUSTED /content/runs/*.log

Wed Nov 11 12:29:36 PM UTC 2026
 12:29:36 up 44 min,  0 user,  load average: 0.45, 0.66, 1.00
100 %, 4719 MiB
total 208
drwxr-xr-x 2 root root   4096 Nov 11 12:23 .
drwxr-xr-x 1 root root   4096 Nov 11 11:54 ..
-rw-r--r-- 1 root root   1023 Nov 11 11:57 cb_lbf.log
-rw-r--r-- 1 root root      0 Nov 11 11:54 cb_rware.log
-rw-r--r-- 1 root root   2791 Nov 11 12:23 cc_lbf.log
-rw-r--r-- 1 root root 193928 Nov 11 12:23 credit_compare_lbf.json
2130 python3 experiments/critic_budget.py --env lbf --agents 4 --window 20 --see
2132 bash -c while kill -0 2130 2>/dev/null; do sleep 30; done; XLA_PYTHON_CLIEN
12793 /bin/bash -c date; uptime; nvidia-smi --query-gpu=utilization.gpu,memory.u
credit": "team", "seed": 0
final_reward_per_step": 0.00726
credit": "team", "seed": 1
final_reward_per_step": 0.00724
credit": "team", "seed": 2
final_reward_per_step": 0.00719
credit": "resp", "seed": 0
final_reward_per_step": 0.0, "a
credit": "resp", "seed": 1
final_reward_per_step": 0.0, "a
credit": "resp", "se

## 10. Reading the credit result: a bug, not a finding

Loads the record and summarises each learning curve.

**Output:** `team` improves from about 0.0013 to 0.0073 reward per step. `resp` drops to 0 within the first 200 updates, below the random starting policy; `dr` is unstable. The diagnosis, confirmed on CPU the same day: the responsibility term was rescaled on its own each batch, so early critic noise outweighed the team signal 35 to 780 times and the policy collapsed onto one action. Fixed in commit `9a01824`. **These credit numbers are void.**

In [4]:
import json, numpy as np
d = json.load(open('/content/runs/credit_compare_lbf.json'))
print(d['summary'])
for k, v in d['curves'].items(): r = np.array(v); print(k, len(r), 'first200', r[:200].mean().round(5), 'last200', r[-200:].mean().round(5), 'max', r.max().round(5), 'nonzero', (r > 0).mean().round(3))
for r in d['runs']: print(r['credit'], r['seed'], r['seconds'], r['branches'])

{'team': {'mean': 0.007235243158259738, 'ci95': [0.007195638101547957, 0.007263997520785779], 'iqm': 0.007240668505352612, 'seeds': 3}, 'resp': {'mean': 0.0, 'ci95': [0.0, 0.0], 'iqm': 0.0, 'seeds': 3}, 'dr': {'mean': 0.00033528646585182284, 'ci95': [0.0, 0.0009928385638340842], 'iqm': 0.00017415364978660364, 'seeds': 3}}
team/0 2000 first200 0.00123 last200 0.00726 max 0.01074 nonzero 0.985
team/1 2000 first200 0.00181 last200 0.00725 max 0.01107 nonzero 0.988
team/2 2000 first200 0.00136 last200 0.0072 max 0.01107 nonzero 0.972
resp/0 2000 first200 0.0 last200 0.0 max 0.00033 nonzero 0.002
resp/1 2000 first200 4e-05 last200 0.0 max 0.00098 nonzero 0.012
resp/2 2000 first200 0.0 last200 0.0 max 0.00065 nonzero 0.001
dr/0 2000 first200 0.00023 last200 1e-05 max 0.00195 nonzero 0.066
dr/1 2000 first200 0.00079 last200 0.00099 max 0.00651 nonzero 0.956
dr/2 2000 first200 0.00104 last200 0.0 max 0.00586 nonzero 0.762
team 0 74.1 0
team 1 46.6 0
team 2 61.9 0
resp 0 433.6 512000
resp 1 385

## 11. Check before mounting Drive (12:46 UTC)

**Output:** Drive not mounted yet. The LBF critic job still running after 50 minutes on its first seed.

In [5]:
!date; uptime; ls /content/drive/MyDrive 2>&1 | head -5; ls -la /content/runs; pgrep -af experiments | cut -c1-80; grep -h -o -e rel_err_test.......................... /content/runs/cb_lbf.log | head

Wed Nov 11 12:46:58 PM UTC 2026
 12:46:58 up  1:01,  0 user,  load average: 1.12, 0.59, 0.67
ls: cannot access '/content/drive/MyDrive': No such file or directory
total 208
drwxr-xr-x 2 root root   4096 Nov 11 12:23 .
drwxr-xr-x 1 root root   4096 Nov 11 11:54 ..
-rw-r--r-- 1 root root   1023 Nov 11 11:57 cb_lbf.log
-rw-r--r-- 1 root root      0 Nov 11 11:54 cb_rware.log
-rw-r--r-- 1 root root   2791 Nov 11 12:23 cc_lbf.log
-rw-r--r-- 1 root root 193928 Nov 11 12:23 credit_compare_lbf.json
2130 python3 experiments/critic_budget.py --env lbf --agents 4 --window 20 --see
2132 bash -c while kill -0 2130 2>/dev/null; do sleep 30; done; XLA_PYTHON_CLIEN
17238 /bin/bash -c date; uptime; ls /content/drive/MyDrive 2>&1 | head -5; ls -l


## 12. Mount Google Drive (approved by Yash)

So that results and checkpoints survive a reset.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 13. The bigger credit run on the fixed code (12:58 UTC)

Pulls commit `9a01824` (the collapse fix), starts a loop that copies results to Drive every 5 minutes, and launches the version-2 credit comparison: 10 seeds x 10,000 updates, written straight to `MyDrive/accteams_runs`.

**What happened next (not in this notebook):** Colab disconnected the runtime around 13:15 to 13:45 UTC for inactivity or maximum duration, then refused a GPU (free-tier usage limit). About seven team-credit seeds reached Drive. The 19 Nov review then changed the critic loss (C1), so those seeds are void too and the reruns use new output names (`tools/colab_resume.sh`).

In [7]:
%%bash
D=/content/drive/MyDrive/accteams_runs; mkdir -p $D
cd /content/accountable-teams && git pull -q origin set-critic && git log --oneline -1
nohup bash -c "while true; do cp -u /content/runs/* $D/ 2>/dev/null; sleep 300; done" > /dev/null 2>&1 &
XLA_PYTHON_CLIENT_MEM_FRACTION=0.3 nohup python experiments/credit_compare.py --env lbf --agents 4 --updates 10000 --seeds 10 --out $D/credit_compare_lbf_v2.json > $D/cc_lbf_v2.log 2>&1 &
sleep 30; ls -la $D; pgrep -af experiments | cut -c1-90; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

9a01824 Fix credit collapse: keep c_i in return units, normalise with GAE, warm up beta
total 203
drwx------ 2 root root   4096 Nov 11 12:58 .
drwx------ 6 root root   4096 Nov 11 12:49 ..
-rw------- 1 root root   1023 Nov 11 12:58 cb_lbf.log
-rw------- 1 root root      0 Nov 11 12:58 cb_rware.log
-rw------- 1 root root   2791 Nov 11 12:58 cc_lbf.log
-rw------- 1 root root    832 Nov 11 12:58 cc_lbf_v2.log
-rw------- 1 root root 193928 Nov 11 12:58 credit_compare_lbf.json
2130 python3 experiments/critic_budget.py --env lbf --agents 4 --window 20 --seeds 3 --out
2132 bash -c while kill -0 2130 2>/dev/null; do sleep 30; done; XLA_PYTHON_CLIENT_MEM_FRAC
20536 python3 experiments/credit_compare.py --env lbf --agents 4 --updates 10000 --seeds 1
100 %, 9317 MiB
